# Rafeeq AI — ML Comparison Pipeline (Google Colab)

This notebook runs the full **Rafeeq ML pipeline** on Google Colab.

## Setup instructions (do this once)

1. Upload your project folder to **Google Drive** so the structure looks like:
   ```
   My Drive/
   └── rafeeq_ml/
       ├── config.py
       ├── 01_eda.py
       ├── 02_preprocessing.py
       ├── 03_model1_arabert.py
       ├── 04_model2_camelbert.py
       ├── 05_model3_lstm.py
       ├── 06_comparison.py
       ├── 07_report.py
       ├── requirements.txt
       └── dataset/
           └── SauDial Dataset.csv
   ```
2. Enable **GPU**: Runtime → Change runtime type → T4 GPU
3. Run each cell from top to bottom.

## Step 1 — Mount Google Drive & set working directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ------------------------------------------------------------------ #
# EDIT THIS PATH if your folder is named differently or nested deeper  #
PROJECT_DIR = '/content/drive/MyDrive/rafeeq_ml'
# ------------------------------------------------------------------ #

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Folder not found: {PROJECT_DIR}\n"
        "Please upload your project to Google Drive and update PROJECT_DIR above."
    )

os.chdir(PROJECT_DIR)
print(f"Working directory set to: {os.getcwd()}")
print("Files found:", os.listdir('.'))

## Step 2 — Install dependencies

In [ ]:
# Install all required packages.
# torch and transformers are already on Colab but we pin versions to
# match requirements.txt.  --quiet suppresses the long pip output.
!pip install -q \
    "torch>=2.0.0" \
    "transformers>=4.36.0" \
    "datasets>=2.16.0" \
    "scikit-learn>=1.3.0" \
    "pandas>=2.0.0" \
    "numpy>=1.24.0" \
    "matplotlib>=3.7.0" \
    "seaborn>=0.13.0" \
    "plotly>=5.18.0" \
    "tqdm>=4.66.0" \
    "arabic-reshaper>=3.0.0" \
    "python-bidi>=0.4.2" \
    "jinja2>=3.1.0" \
    "openpyxl>=3.1.0" \
    "colorama>=0.4.6"

print("All packages installed.")

## Step 3 — Verify dataset & GPU

In [ ]:
import torch

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found — training will use CPU (slower).")
    print("Tip: Runtime → Change runtime type → T4 GPU")

# Check dataset
dataset_path = os.path.join(PROJECT_DIR, 'dataset', 'SauDial Dataset.csv')
if os.path.exists(dataset_path):
    print(f"Dataset found: {dataset_path}")
else:
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_path}\n"
        "Make sure 'SauDial Dataset.csv' is inside the 'dataset/' folder."
    )

---
## Pipeline Step 1 — Exploratory Data Analysis

In [ ]:
%run 01_eda.py

In [ ]:
# Preview the EDA charts inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

eda_charts = sorted(glob.glob('outputs/eda/*.png'))
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, path in zip(axes.flat, eda_charts):
    ax.imshow(mpimg.imread(path))
    ax.set_title(os.path.basename(path), fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(eda_charts):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Pipeline Step 2 — Data Preprocessing

In [ ]:
%run 02_preprocessing.py

In [ ]:
# Quick sanity check on the produced splits
import pandas as pd

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'outputs/data/{split}.csv', encoding='utf-8-sig')
    print(f"{split:>5}: {len(df):,} rows — intents: {df['intent'].nunique()}")

## Pipeline Step 3 — Model 1: Whisper ASR + AraBERT

> This step downloads the AraBERT model from HuggingFace (~500 MB) and fine-tunes it.
> Falls back to TF-IDF + Logistic Regression if the download fails.

In [ ]:
%run 03_model1_arabert.py

## Pipeline Step 4 — Model 2: Wav2Vec2 ASR + CAMeL-BERT

> Downloads the CAMeL-BERT model (~400 MB) and fine-tunes it.
> Falls back to TF-IDF + Logistic Regression if needed.

In [ ]:
%run 04_model2_camelbert.py

## Pipeline Step 5 — Model 3: DeepSpeech ASR + BiLSTM

> Trains a from-scratch BiLSTM with attention in PyTorch. No download required.

In [ ]:
%run 05_model3_lstm.py

## Pipeline Step 6 — Comparison Charts

In [ ]:
%run 06_comparison.py

In [ ]:
# Preview all comparison charts inline
comparison_charts = sorted(glob.glob('outputs/comparison/*.png'))
cols = 2
rows = -(-len(comparison_charts) // cols)  # ceiling division
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 6))
for ax, path in zip(axes.flat, comparison_charts):
    ax.imshow(mpimg.imread(path))
    ax.set_title(os.path.basename(path), fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(comparison_charts):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Pipeline Step 7 — Generate HTML Report

In [ ]:
%run 07_report.py

In [ ]:
# Display the HTML report directly inside the notebook
from IPython.display import HTML, display

report_path = 'outputs/report/rafeeq_ml_report.html'
if os.path.exists(report_path):
    with open(report_path, encoding='utf-8') as f:
        display(HTML(f.read()))
else:
    print("Report not found — check that 07_report.py ran without errors.")

## Download outputs

Since the `outputs/` folder is inside Google Drive, all files are already saved there automatically.
You can also download individual files using the cell below.

In [ ]:
from google.colab import files

# Download the HTML report
files.download('outputs/report/rafeeq_ml_report.html')

In [ ]:
# Optional: download all metrics JSON files
import json

for model in ['model1', 'model2', 'model3']:
    path = f'outputs/results/{model}_metrics.json'
    if os.path.exists(path):
        with open(path) as f:
            m = json.load(f)
        label = m.get('model_label', model)
        print(f"[{label}]  accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
    else:
        print(f"{path} — not found (training may have been skipped)")